<a href="https://colab.research.google.com/github/abo7mied/nlp-for-fundamental-analysis/blob/Abd/ABD_branch_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import re
import math
import random
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

seed = 42
random.seed(seed)
torch.manual_seed(seed)

device = "cpu"


In [2]:
# settings
max_len = 32
max_vocab_size = 5000
batch_size = 32
num_epochs = 2
lr = 1e-4
mlm_probability = 0.15
d_model = 32
d_internal = 64
num_layers = 2

In [3]:
texts = [
    "Aramco showed strong growth this quarter.",
    "Tesla reported strong revenue this quarter",
    "Tesla reported strong revenue after demand increased.",
    "Apple reported weak guidance despite strong revenue.",
    "The company beat earnings expectations this quarter.",
    "The stock declined after the company warned about margin pressure.",
    "Investors reacted positively to revenue growth.",
    "The firm missed analyst expectations and shares fell.",
    "Strong earnings helped the company recover.",
    "Weak demand created pressure on future revenue."
] * 120

random.shuffle(texts)
cut = int(0.8 * len(texts))
train_texts = texts[:cut]
val_texts = texts[cut:]

print(len(train_texts), len(val_texts))
print(train_texts[0])


960 240
Weak demand created pressure on future revenue.


In [10]:
PAD = "[PAD]"
UNK = "[UNK]"
CLS = "[CLS]"
SEP = "[SEP]"
MASK = "[MASK]"
special_tokens = [PAD, UNK, CLS, SEP, MASK]

def basic_tokenize(text):
    text = str(text)
    text = re.sub(r"\[mask\]", " [MASK] ", text, flags=re.IGNORECASE)
    text = text.lower()
    text = text.replace("[mask]", "[MASK]")
    pattern = r"\[MASK\]|\$?\d+(?:\.\d+)?%?|[a-z]+(?:-[a-z]+)*|[.,!?;:]"
    return re.findall(pattern, text)

def build_vocab(texts):
    counter = Counter()
    for text in texts:
        counter.update(basic_tokenize(text))

    stoi = {}
    for token in special_tokens:
        stoi[token] = len(stoi)

    for token, count in counter.most_common(max_vocab_size - len(special_tokens)):
        if token not in stoi:
            stoi[token] = len(stoi)

    itos = {i: token for token, i in stoi.items()}
    return stoi, itos

stoi, itos = build_vocab(train_texts)
print("vocab size:", len(stoi))
print(stoi)


vocab size: 48
{'[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3, '[MASK]': 4, '.': 5, 'strong': 6, 'revenue': 7, 'the': 8, 'this': 9, 'quarter': 10, 'reported': 11, 'company': 12, 'tesla': 13, 'after': 14, 'demand': 15, 'growth': 16, 'earnings': 17, 'expectations': 18, 'pressure': 19, 'weak': 20, 'increased': 21, 'aramco': 22, 'showed': 23, 'beat': 24, 'investors': 25, 'reacted': 26, 'positively': 27, 'to': 28, 'helped': 29, 'recover': 30, 'firm': 31, 'missed': 32, 'analyst': 33, 'and': 34, 'shares': 35, 'fell': 36, 'stock': 37, 'declined': 38, 'warned': 39, 'about': 40, 'margin': 41, 'created': 42, 'on': 43, 'future': 44, 'apple': 45, 'guidance': 46, 'despite': 47}


In [11]:
def encode_text(text):
    tokens = basic_tokenize(text)
    tokens = [CLS] + tokens[:max_len - 2] + [SEP]

    ids = []
    for token in tokens:
        ids.append(stoi.get(token, stoi[UNK]))

    attn_mask = [1] * len(ids)

    # TODO: why do we mask non-existing words?
    while len(ids) < max_len:
        ids.append(stoi[PAD])
        attn_mask.append(0)

    return ids, attn_mask

class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        ids, attn_mask = encode_text(self.texts[index])
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn_mask, dtype=torch.long)
        }

train_loader = DataLoader(TextDataset(train_texts), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TextDataset(val_texts), batch_size=batch_size, shuffle=False)

batch = next(iter(train_loader))
print(batch["input_ids"].shape)


torch.Size([32, 32])


In [12]:
class TransformerLayer(nn.Module):
    def __init__(self, d_model, d_internal):
        super().__init__()
        self.q = nn.Linear(d_model, d_internal)
        self.k = nn.Linear(d_model, d_internal)
        self.v = nn.Linear(d_model, d_internal)
        self.attn_out = nn.Linear(d_internal, d_model)

        self.fc1 = nn.Linear(d_model, d_internal)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(d_internal, d_model)

        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def attention(self, x, attention_mask):
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(q.shape[-1])

        ## NOTE: We need to understand this
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(1) # To allow broadcasting
            scores = scores.masked_fill(mask == 0, -1e9)

        weights = torch.softmax(scores, dim=-1)
        out = weights @ v
        out = self.attn_out(out)
        return out

    def ffn(self, x):
        return self.fc2(self.relu(self.fc1(x)))

    def forward(self, x, attention_mask):
        z = self.ln1(x + self.attention(x, attention_mask))
        y = self.ln2(z + self.ffn(z))
        return y


## GPT-heavy

In [13]:
class TransformerLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model, d_internal, num_layers, max_len):
        super().__init__()
        self.vocab_size = vocab_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)

        layers = []
        for i in range(num_layers):
            layers.append(TransformerLayer(d_model, d_internal))
        self.layers = nn.ModuleList(layers)

        self.classifier = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, attention_mask):
        b, t = input_ids.shape
        pos = torch.arange(t, device=input_ids.device).unsqueeze(0)
        x = self.token_embedding(input_ids) + self.position_embedding(pos)

        for layer in self.layers:
            x = layer(x, attention_mask)

        logits = self.classifier(x)
        return logits

model = TransformerLanguageModel(
    vocab_size=len(stoi),
    d_model=d_model,
    d_internal=d_internal,
    num_layers=num_layers,
    max_len=max_len
).to(device)

print(model.__class__.__name__)


TransformerLanguageModel


In [14]:
def make_mlm_batch(input_ids, attention_mask):
    labels = input_ids.clone()
    masked_ids = input_ids.clone()

    special = (input_ids == stoi[PAD]) | (input_ids == stoi[CLS]) | (input_ids == stoi[SEP])
    rand = torch.rand(input_ids.shape, device=input_ids.device)
    masked_positions = (rand < mlm_probability) & (~special) & (attention_mask == 1)

    for row in range(input_ids.shape[0]):
        if not masked_positions[row].any():
            possible = torch.where((~special[row]) & (attention_mask[row] == 1))[0]
            if len(possible) > 0:
                picked = possible[torch.randint(len(possible), (1,), device=input_ids.device)]
                masked_positions[row, picked] = True

    labels[~masked_positions] = -100
    masked_ids[masked_positions] = stoi[MASK]
    return masked_ids, labels


In [15]:
def get_metrics(logits, labels, loss_value):
    masked = labels != -100
    selected_logits = logits[masked]
    selected_labels = labels[masked]

    if selected_labels.numel() == 0:
        return 0.0, 0.0, None

    preds = selected_logits.argmax(dim=-1)
    top1 = (preds == selected_labels).float().mean().item()

    k = min(5, selected_logits.shape[-1])
    topk = selected_logits.topk(k, dim=-1).indices
    top5 = (topk == selected_labels.unsqueeze(-1)).any(dim=-1).float().mean().item()

    ppl = math.exp(loss_value) if loss_value < 20 else None
    return top1, top5, ppl


In [17]:
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

best_val_loss = float("inf")
history = []

def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0
    total_top1 = 0
    total_top5 = 0
    steps = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        masked_ids, labels = make_mlm_batch(input_ids, attention_mask)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(masked_ids, attention_mask)
            loss = loss_fn(logits.reshape(-1, len(stoi)), labels.reshape(-1)) # UNDERSTAND

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # UNDERSTAND
                optimizer.step()

        top1, top5, ppl = get_metrics(logits, labels, loss.item())
        total_loss += loss.item()
        total_top1 += top1
        total_top5 += top5
        steps += 1

    return total_loss / steps, total_top1 / steps, total_top5 / steps


## Continue from here

In [18]:
for epoch in range(num_epochs):
    train_loss, train_top1, train_top5 = run_epoch(train_loader, training=True)
    val_loss, val_top1, val_top5 = run_epoch(val_loader, training=False)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_top1": train_top1,
        "val_top1": val_top1,
        "train_top5": train_top5,
        "val_top5": val_top5
    })

    print("epoch", epoch + 1)
    print("train loss", round(train_loss, 4), "top1", round(train_top1, 4), "top5", round(train_top5, 4))
    print("val loss", round(val_loss, 4), "top1", round(val_top1, 4), "top5", round(val_top5, 4))

    if val_loss < best_val_loss:
        best_val_loss = val_loss

epoch 1
train loss 3.9499 top1 0.0185 top5 0.1698
val loss 3.8078 top1 0.0658 top5 0.19
epoch 2
train loss 3.7737 top1 0.0709 top5 0.2034
val loss 3.6772 top1 0.0956 top5 0.246


In [19]:
def predict_masked_words(text, top_k=5):
    model.eval()
    ids, attn_mask = encode_text(text)
    input_ids = torch.tensor([ids], dtype=torch.long).to(device)
    attention_mask = torch.tensor([attn_mask], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)

    mask_positions = torch.where(input_ids[0] == stoi[MASK])[0]
    results = []

    for pos in mask_positions:
        values, token_ids = torch.topk(logits[0, pos], k=top_k)
        row = []
        for token_id in token_ids:
            row.append(itos[token_id.item()])
        results.append(row)

    return results

print(predict_masked_words("Ahmed is a [MASK] math student.", top_k=5))

# TODO: make prediction print a completed sentence, not just candidate words


[['the', '.', 'about', 'recover', 'warned']]
